# 03 · Cohort Retention

**Goal:** find out how many customers come back to buy in the months after their first purchase.

**Source:** `gold.fact_sales` (orders with a valid date)

**Method:** a customer's *cohort* is the month of their first purchase. *Cohort index* is the number of months since that purchase. Retention for month *k* = customers active in month *k* ÷ cohort size.

**Output:** retention matrix, cohort sizes, mature-cohort summary and a heatmap in `output/`

## 1. Setup

In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import config

sns.set_style("white")
os.makedirs("output", exist_ok=True)

## 2. Load orders

In [8]:
engine = create_engine(config.get_conn_url())
df = pd.read_sql(
    """
    SELECT order_number, order_date, customer_key
    FROM gold.fact_sales
    WHERE order_date IS NOT NULL
    """,
    engine,
)
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

## 3. One customer-month observation

A customer who orders several times in one month is counted once for that month.

In [9]:
df["order_month"] = df["order_date"].dt.to_period("M")

customer_month = df[["customer_key", "order_month"]].drop_duplicates().copy()

customer_month["cohort_month"] = customer_month.groupby("customer_key")[
    "order_month"
].transform("min")
customer_month["cohort_index"] = (
    customer_month["order_month"] - customer_month["cohort_month"]
).apply(lambda x: x.n)

print(customer_month.head())

   customer_key order_month cohort_month  cohort_index
0         10769     2010-12      2010-12             0
1         17390     2010-12      2010-12             0
2         14864     2010-12      2010-12             0
3          3502     2010-12      2010-12             0
4             4     2010-12      2010-12             0


## 4. Cohort retention matrix

In [10]:
cohort_counts = (
    customer_month.groupby(["cohort_month", "cohort_index"])["customer_key"]
    .nunique()
    .unstack(fill_value=0)
)
cohort_sizes = cohort_counts[0]
retention = cohort_counts.divide(cohort_sizes, axis=0)
print((retention.iloc[:, :12] * 100).round(1))

cohort_index     0    1    2    3    4    5    6     7     8     9     10  \
cohort_month                                                                
2010-12       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-01       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-02       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-03       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-04       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-05       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-06       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-07       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-08       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-09       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   
2011-10       100.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   0.0   0.0   

Month 0 is 100% by definition. For most cohorts, retention in the following months is close to 0%.

## 5. Mature-cohort retention

For month *k*, only cohorts old enough to be observed for *k* months are included. Younger cohorts would otherwise show 0% simply because they have not had time to return.

In [28]:
max_observed_index = retention.columns.max()
mature_rows = {}
for k in range(1, min(12, int(max_observed_index)) + 1):
    eligible = retention.index[retention.index <= retention.index.max() - k]
    if len(eligible) > 0:
        mature_rows[k] = {
            "eligible_cohorts": len(eligible),
            "mean_retention_pct": round(retention.loc[eligible, k].mean() * 100, 1),
            "median_retention_pct": round(retention.loc[eligible, k].median() * 100, 1),
        }
mature_summary = pd.DataFrame(mature_rows).T
print(mature_summary)

    eligible_cohorts  mean_retention_pct  median_retention_pct
1               37.0                 1.0                   0.0
2               36.0                 1.2                   0.0
3               35.0                 1.3                   0.0
4               34.0                 1.3                   0.0
5               33.0                 1.6                   0.0
6               32.0                 2.0                   0.0
7               31.0                 2.2                   0.0
8               30.0                 2.6                   0.0
9               29.0                 2.3                   0.0
10              28.0                 3.0                   0.2
11              27.0                 5.6                   0.4
12              26.0                 5.9                   1.2


Mean retention is 1.0% at month 1 and 5.9% at month 12. The median stays at 0.0% up to month 9 and reaches 1.2% at month 12.

## 6. Heatmap

In [24]:
max_cols = min(12, retention.shape[1])
plot_data = retention.iloc[:, :max_cols] * 100

plt.figure(figsize=(14, 8))
sns.heatmap(
    plot_data,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    vmin=0,
    vmax=100,
    cbar_kws={"label": "Retention %"},
)
plt.title("Monthly Cohort Retention Rate (%)")
plt.xlabel("Months Since First Purchase")
plt.ylabel("Cohort")
plt.tight_layout()
plt.savefig("output/06_cohort_retention_heatmap_v2.png", dpi=120)
plt.close()

![Monthly cohort retention heatmap](output/06_cohort_retention_heatmap_v2.png)

## 7. Headline retention numbers

In [29]:
month_1 = (
    mature_summary.loc[1, "median_retention_pct"]
    if 1 in mature_summary.index
    else np.nan
)
month_3 = (
    mature_summary.loc[3, "median_retention_pct"]
    if 3 in mature_summary.index
    else np.nan
)
month_6 = (
    mature_summary.loc[6, "median_retention_pct"]
    if 6 in mature_summary.index
    else np.nan
)

print(f"Median mature-cohort Month-1 retention: {month_1}%")
print(f"Median mature-cohort Month-3 retention: {month_3}%")
print(f"Median mature-cohort Month-6 retention: {month_6}%")

Median mature-cohort Month-1 retention: 0.0%
Median mature-cohort Month-3 retention: 0.0%
Median mature-cohort Month-6 retention: 0.0%


Median mature-cohort retention is 0.0% at months 1, 3 and 6: most customers do not buy again in the months after their first purchase.

## 8. Save outputs

In [30]:
retention.to_csv("output/cohort_retention_matrix_v2.csv")
cohort_sizes.to_csv("output/cohort_sizes_v2.csv")
mature_summary.to_csv("output/mature_cohort_retention_summary.csv")
print("Cohort analysis complete.")

Cohort analysis complete.
